In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [3]:
spark = SparkSession.builder \
    .appName("SmartHomeEnergyAggregation") \
    .getOrCreate()

In [5]:
df = spark.read.csv(
    "/content/energy_usage_cleaned.csv",
    header=True,
    inferSchema=True
)

In [6]:
df = df.filter(col("energy_kwh") > 0)

df = df.withColumn("reading_hour", hour(col("timestamp")))

df = df.withColumn(
    "usage_period",
    when((col("reading_hour") >= 6) & (col("reading_hour") < 10), "Peak")
    .when((col("reading_hour") >= 18) & (col("reading_hour") < 23), "Peak")
    .otherwise("Off-Peak")
)

In [9]:
device_period_summary = df.groupBy("device_name", "usage_period") \
    .agg(
        round(sum("energy_kwh"), 3).alias("total_energy_kwh"),
        round(avg("energy_kwh"), 3).alias("avg_energy_kwh")
    ) \
    .orderBy("device_name", "usage_period")

In [10]:
print("\nPeak vs Off-Peak usage by device:")
device_period_summary.show(30, truncate=False)


Peak vs Off-Peak usage by device:
+----------------+------------+----------------+--------------+
|device_name     |usage_period|total_energy_kwh|avg_energy_kwh|
+----------------+------------+----------------+--------------+
|Air Conditioner |Off-Peak    |39.631          |1.468         |
|Air Conditioner |Peak        |27.254          |1.434         |
|Air Purifier    |Off-Peak    |0.95            |0.106         |
|Air Purifier    |Peak        |0.89            |0.099         |
|Ceiling Fan     |Off-Peak    |1.527           |0.139         |
|Ceiling Fan     |Peak        |1.415           |0.142         |
|Desktop Computer|Off-Peak    |5.799           |0.483         |
|Desktop Computer|Peak        |3.317           |0.474         |
|Microwave Oven  |Off-Peak    |6.13            |0.383         |
|Microwave Oven  |Peak        |2.856           |0.357         |
|Printer         |Off-Peak    |0.488           |0.03          |
|Printer         |Peak        |0.249           |0.031         |
|Refr

In [11]:
device_totals = df.groupBy("device_name") \
    .agg(round(sum("energy_kwh"), 3).alias("total_energy_kwh")) \
    .orderBy(col("total_energy_kwh").desc())

print("\nAll devices ranked by total energy usage:")
device_totals.show(20, truncate=False)


All devices ranked by total energy usage:
+----------------+----------------+
|device_name     |total_energy_kwh|
+----------------+----------------+
|Air Conditioner |66.884          |
|Refrigerator    |9.437           |
|Desktop Computer|9.116           |
|Microwave Oven  |8.986           |
|Smart TV        |6.922           |
|Ceiling Fan     |2.942           |
|Table Lamp      |1.934           |
|Air Purifier    |1.84            |
|Printer         |0.736           |
+----------------+----------------+



In [12]:
top_devices = device_totals.filter(col("total_energy_kwh") > 5).limit(5)

print("\nTop energy-consuming devices (filtered, >5 kWh total):")
top_devices.show(truncate=False)


Top energy-consuming devices (filtered, >5 kWh total):
+----------------+----------------+
|device_name     |total_energy_kwh|
+----------------+----------------+
|Air Conditioner |66.884          |
|Refrigerator    |9.437           |
|Desktop Computer|9.116           |
|Microwave Oven  |8.986           |
|Smart TV        |6.922           |
+----------------+----------------+



In [13]:
device_period_summary.toPandas().to_csv(
    "/content/device_peak_offpeak_summary.csv", index=False
)

In [14]:
top_devices.toPandas().to_csv(
    "/content/top_devices_by_usage.csv", index=False
)

In [15]:
spark.stop()